In [ ]:
!pip uninstall -y mcp fastmcp langchain-mcp-adapters langgraph langchain-google-genai
!pip install -q "mcp>=1.0.0,<2.0.0" mcp-types langgraph langchain-google-genai nest_asyncio

Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 25.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency c

In [ ]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata
# LangChain & LangGraph Imports
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI
nest_asyncio.apply()
# 1. Setup Gemini API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key")
# 2. Initialize Gemini LLM (Use valid model name)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
# -------------------------------------------------------------
# 3. Local Real-World Tools (MCP Primitive Logic)
# -------------------------------------------------------------
CSV_FILE = "expenses.csv"
def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Item", "Amount", "Category"])
def add_expense(item: str, amount: float, category: str) -> str:
    """Logs a new expense with item, amount, and category into CSV."""
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([item, amount, category])
    return f"Successfully logged expense: {item} - ${amount} ({category})"
def get_expenses() -> str:
    """Retrieves all logged expenses from the CSV file."""
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        reader = csv.reader(f)
        rows = list(reader)
    if len(rows) <= 1:
        return "No expenses recorded yet."
    return "\n".join([", ".join(row) for row in rows])
# -------------------------------------------------------------
# 4. Wrap Tools as LangChain Structured Tools for LangGraph
# -------------------------------------------------------------
mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs a new expense with item, amount, and category."
    ),
    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Retrieves all logged expenses from the expense tracker."
    )
]
# -------------------------------------------------------------
# 5. Build & Execute LangGraph Agent
# -------------------------------------------------------------
agent = create_react_agent(llm, mcp_tools)
async def run_agentic_workflow():
    print("--- Task 1: Log an expense ---")
    prompt_1 = "I bought a pizza for $12.50. Category is Food."
    response_1 = await agent.ainvoke({"messages": [("user", prompt_1)]})
    # Correct attribute inspection for LangChain BaseMessage objects
    for msg in response_1["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")
    print("\n--- Task 2: Retrieve records ---")
    prompt_2 = "Show me all expenses logged so far."
    response_2 = await agent.ainvoke({"messages": [("user", prompt_2)]})
    for msg in response_2["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")
# Run Execution Loop
asyncio.run(run_agentic_workflow())

/tmp/ipykernel_3248/1868270702.py:58: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Task 1: Log an expense ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': "I've logged your expense: **pizza** for **$12.50** under **Food**.", 'extras': {'signature': 'EtIBCs8BARFNMg+8sRNeKr7xdPS5hTJz76ZFBNAfyDPPduvraOM+6Rv7MX6rYJ91x9pIXEWNsy8Vqmit8Aq+gSCplSS145qYQJa82li26uA4/rL0YFZdQ+ZaYMeMqxhqLwvDIAvB3nlPW3yD9sXjGixEXzvDxnCcVkPAZYONxfNN9Rtqeg116QBckYbpGsNHd948ONN2xf7+CbheutFDXAWwKhWH4IcxQhG4beNlJHCVhg+F/iXPlipNrwSSpNZhMgC902QpYPwFhvqUJnGXs7UBblkj'}}]

--- Task 2: Retrieve records ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': 'Here are all the expenses logged so far:\n\n* **Item:** Pizza\n* **Amount:** $12.50\n* **Category:** Food', 'extras': {'signature': 'EtEBCs4BARFNMg84x3Ih5CoID4ze5/yf1lfsJOyPEXg3trypSREtacQ/JzbsPLNzkqtvqBppM38M06abLMnu76qoB/qQ8pGe7YIXuD1P66ZBSOeqvOGHhXbQzxAxPwG/lm75KmVdqvwAAXAIqgsZV+JYFPu0TG/22eo1UcY52TC6hEKOnyN8esprP2/+i3BObCtQZVrS84R9bxlJuSOS8UR7NlI7lbNLUaH2VVLP3fGiaEj+Pu3896px45yyhaATlVy7vVmnlFeuEgyLkb4QFEC5NwE='}}]


In [ ]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

# 1. API & Model Setup
os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key")
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# 2. Expense Tools
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            csv.writer(f).writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        csv.writer(f).writerow([item, amount, category])
    return f"Logged: {item} - ${amount} ({category})"

def get_expenses() -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        rows = list(csv.reader(f))
    return "No expenses." if len(rows) <= 1 else "\n".join([", ".join(r) for r in rows[1:]])

# 3. Wrap Tools & Build Agent
mcp_tools = [
    StructuredTool.from_function(func=add_expense, name="add_expense", description="Logs an expense."),
    StructuredTool.from_function(func=get_expenses, name="get_expenses", description="Gets logged expenses.")
]

agent = create_agent(llm, mcp_tools)

# 4. Helper to extract raw text string
def get_clean_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join([b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text"])
    return str(content)

# 5. Execution
async def run_agentic_workflow():
    res1 = await agent.ainvoke({"messages": [("user", "I bought a pizza for $12.50. Category is Food.")]})
    print(get_clean_text(res1["messages"][-1].content))

    res2 = await agent.ainvoke({"messages": [("user", "Show me all expenses logged so far.")]})
    print(get_clean_text(res2["messages"][-1].content))

asyncio.run(run_agentic_workflow())

I've logged your expense: $12.50 for pizza under the Food category.
Here are all the expenses logged so far:

1. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food
2. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food

**Total:** $25.00
